# IAN 620 · Module 9 Assignment
## Build a Small Agentic RAG Assistant with LangChain + LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ali2449/IAN-620/blob/main/IAN_620_Module_9_Assignment_Agentic_RAG_Starter.ipynb)

This notebook is a **starter notebook** for the Module 9 assignment.

You will work with a larger knowledge-base CSV in which each row contains a fairly long document. You should **chunk the documents before creating embeddings**.

### Main workflow

**CSV → LangChain Documents → Chunking → Embeddings → Chroma → Tools → LangGraph → Agentic RAG**

The notebook gives you a simple working baseline. Your assignment is to improve the important parts and explain your decisions.

> Running the notebook without making the required changes is not enough for the assignment.

## What you will complete

You will:

1. Load the larger Module 9 dataset from GitHub.
2. Compare at least **two chunking settings**.
3. Choose a final chunk size and overlap.
4. Build local embeddings and a Chroma vector database.
5. Create a knowledge-base search tool.
6. Use a simple order lookup tool as a second source.
7. Build shared agent state.
8. Route the question.
9. Check whether retrieved evidence is useful.
10. Rewrite and retrieve again when evidence is weak.
11. Build the workflow with LangGraph.
12. Test at least four questions.
13. Add a short reflection.

The goal is not to build a large application. Keep your code readable and make the control flow easy to follow.

## 0. Install the libraries

Run this cell once in Colab.

No paid LLM API is required. The notebook uses:

- LangChain document and retrieval components
- LangGraph for the workflow
- Sentence Transformers for local embeddings
- Chroma for the vector database
- a small local Qwen model for final answer generation

In [ ]:
%pip install -q -U \
    langchain \
    langgraph \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    chromadb

## 1. Imports

In [ ]:
import re
import operator
from typing import TypedDict, Annotated

import pandas as pd

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## 2. Load the larger knowledge base

The dataset contains long policy and support documents. Each row is a **source document**, not a chunk.

Do not create one embedding per CSV row and stop there. The purpose of this assignment is to practice chunking.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base_Large.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", list(df.columns))

df.head(3)

### Inspect document length

Before chunking, look at how large the source documents are.

In [ ]:
df["word_count"] = df["document_text"].str.split().str.len()

df[["document_id", "title", "category", "word_count"]].head(10)

In [ ]:
print("Average words per document:", round(df["word_count"].mean(), 1))
print("Shortest document:", df["word_count"].min())
print("Longest document:", df["word_count"].max())

## 3. Convert rows to LangChain Documents

The long text goes into `page_content`.

Important source information should stay in `metadata` so that it remains attached after chunking.

In [ ]:
documents = []

for _, row in df.iterrows():
    doc = Document(
        page_content=row["document_text"],
        metadata={
            "document_id": row["document_id"],
            "title": row["title"],
            "category": row["category"],
            "department": row["department"],
            "region": row["region"],
            "audience": row["audience"],
            "primary_channel": row["primary_channel"],
            "last_updated": row["last_updated"],
        },
    )
    documents.append(doc)

print("LangChain Documents:", len(documents))
print()
print(documents[0].metadata)
print()
print(documents[0].page_content[:500])

## 4. Chunking experiment

Chunking affects retrieval.

A chunk that is too large can contain too many unrelated ideas. A chunk that is too small can lose useful context.

Start by comparing two settings.

In [ ]:
def split_documents(source_documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(source_documents)


chunks_a = split_documents(
    documents,
    chunk_size=500,
    chunk_overlap=75,
)

chunks_b = split_documents(
    documents,
    chunk_size=1000,
    chunk_overlap=150,
)

print("Setting A chunks:", len(chunks_a))
print("Setting B chunks:", len(chunks_b))

### Look at actual chunks

Notice that LangChain keeps the parent-document metadata on the chunks.

In [ ]:
sample_chunk = chunks_a[0]

print(sample_chunk.metadata)
print()
print(sample_chunk.page_content)

### Assignment task — choose your final chunking settings

Change the values below after examining both approaches.

You should briefly explain in a Markdown cell:

- which settings you selected,
- why you selected them,
- and what difference you noticed between the two chunking approaches.

In [ ]:
# TODO: choose your own final values after the comparison

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

chunks = split_documents(
    documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

print("Final number of chunks:", len(chunks))

### Your chunking explanation

**TODO:** Replace this text with your short explanation.

- Setting A:
- Setting B:
- Final setting:
- Why I chose it:

## 5. Create local embeddings

We will use a small Sentence Transformers model.

This runs locally and does not require an API key.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## 6. Store chunks in Chroma

Every chunk is embedded and stored in the vector database.

The metadata from the original document is preserved.

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="ian620_module9_assignment",
)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector database ready.")

## 7. Test retrieval before building the agent

Always test the retriever by itself first.

If retrieval is poor, an agent will not magically fix the underlying data problem.

In [ ]:
test_question = "How long do I have to return an opened computer?"

results = retriever.invoke(test_question)

for i, doc in enumerate(results, start=1):
    print("=" * 70)
    print("Result", i)
    print("Title:", doc.metadata.get("title"))
    print("Category:", doc.metadata.get("category"))
    print("Region:", doc.metadata.get("region"))
    print()
    print(doc.page_content[:700])

## 8. LangChain tool 1 — search the knowledge base

The first tool searches Chroma and returns the retrieved chunks with visible source information.

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the support knowledge base for policy and procedure information."""

    docs = retriever.invoke(query)

    if not docs:
        return "No relevant knowledge-base evidence was found."

    parts = []

    for i, doc in enumerate(docs, start=1):
        parts.append(
            f"""SOURCE {i}
document_id: {doc.metadata.get("document_id")}
title: {doc.metadata.get("title")}
category: {doc.metadata.get("category")}
region: {doc.metadata.get("region")}

{doc.page_content}
"""
        )

    return "\n\n".join(parts)

In [ ]:
print(search_knowledge_base.invoke(
    "What is the normal return window for electronics?"
)[:2000])

## 9. LangChain tool 2 — simple order lookup

A real company might use a database or API.

For this assignment, we use a small Python dictionary so we can focus on the agent workflow.

In [ ]:
orders = {
    "1042": {
        "status": "delivered",
        "days_since_delivery": 20,
        "product": "laptop",
        "region": "US - East",
    },
    "2088": {
        "status": "processing",
        "days_since_delivery": None,
        "product": "camera",
        "region": "US - West",
    },
    "3310": {
        "status": "delivered",
        "days_since_delivery": 5,
        "product": "headphones",
        "region": "US - Central",
    },
    "5502": {
        "status": "delivered",
        "days_since_delivery": 8,
        "product": "tablet",
        "region": "Canada",
    },
}


@tool
def lookup_order(order_id: str) -> str:
    """Look up basic information for an order number."""

    order = orders.get(order_id)

    if order is None:
        return f"Order {order_id} was not found."

    return (
        f"Order {order_id}: "
        f"status={order['status']}, "
        f"days_since_delivery={order['days_since_delivery']}, "
        f"product={order['product']}, "
        f"region={order['region']}"
    )

In [ ]:
print(lookup_order.invoke("1042"))

### Assignment task — modify the order data

Add at least **one additional order** to the `orders` dictionary.

Use that order in one of your final test questions.

## 10. Load a small local LLM

This section uses **Qwen3-0.6B**.

It is small enough for a teaching notebook and does not require an API key.

If you are using a slower CPU environment, the first model load and generation can take longer than retrieval.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=180,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(
    pipeline=text_generator
)

print("Local LLM ready.")

## 11. Define the LangGraph state

State is the visible working record of the workflow.

It is not hidden model reasoning.

We will keep the user question, current retrieval query, evidence, order result, attempt count, final answer, and a simple execution trace.

In [ ]:
class AgentState(TypedDict):
    question: str
    query: str
    needs_order: bool
    order_id: str
    evidence: str
    order_result: str
    evidence_ok: bool
    attempts: int
    answer: str
    trace: Annotated[list[str], operator.add]

## 12. Helper: detect an order number

This is a deterministic task, so normal Python is enough.

We do not need an LLM to find a simple four-digit order number.

In [ ]:
def extract_order_id(text):
    match = re.search(r"\b\d{4}\b", text)

    if match:
        return match.group()

    return ""

## 13. Node: route the question

This first baseline route is intentionally simple.

If the question contains an order number, the workflow retrieves policy information **and** looks up the order.

Otherwise, it only retrieves knowledge-base evidence.

### Assignment task

Modify this routing rule in a small but meaningful way.

For example, you could:

- recognize a shipping/tracking question,
- recognize an account-access question,
- or use keywords to choose a category-specific retrieval query.

Keep the routing understandable.

In [ ]:
def route_node(state: AgentState):
    question = state["question"]

    order_id = extract_order_id(question)
    needs_order = bool(order_id)

    route = "knowledge base + order lookup" if needs_order else "knowledge base"

    return {
        "query": question,
        "needs_order": needs_order,
        "order_id": order_id,
        "trace": [f"route -> {route}"],
    }

## 14. Node: retrieve knowledge-base evidence

In [ ]:
def retrieve_node(state: AgentState):
    evidence = search_knowledge_base.invoke(state["query"])

    return {
        "evidence": evidence,
        "trace": [f"retrieve -> query: {state['query']}"],
    }

## 15. Node: look up the order when needed

In [ ]:
def order_node(state: AgentState):
    result = lookup_order.invoke(state["order_id"])

    return {
        "order_result": result,
        "trace": [f"order lookup -> {state['order_id']}"],
    }

## 16. Node: check the evidence

This baseline uses a very simple rule.

### Assignment task

Improve this function so it uses at least **two signals**.

Possible signals:

- amount of retrieved text,
- expected keywords,
- matching category,
- matching region,
- whether an order was found,
- whether the question terms appear in the evidence.

Keep it simple. You do not need an LLM grader.

In [ ]:
def grade_node(state: AgentState):
    evidence = state["evidence"]

    enough_text = len(evidence) > 600

    order_ok = True
    if state["needs_order"]:
        order_ok = "not found" not in state["order_result"].lower()

    evidence_ok = enough_text and order_ok

    return {
        "evidence_ok": evidence_ok,
        "trace": [f"grade -> evidence_ok={evidence_ok}"],
    }

## 17. Node: rewrite the retrieval query

If evidence is weak, the workflow should not immediately give up.

It can refine the search query and retrieve again.

### Assignment task

Improve this query-rewrite strategy.

Your rewrite should preserve the original intent while making retrieval more specific.

In [ ]:
def rewrite_node(state: AgentState):
    new_query = (
        state["question"]
        + " policy eligibility requirements exceptions region"
    )

    return {
        "query": new_query,
        "attempts": state["attempts"] + 1,
        "trace": [f"rewrite -> {new_query}"],
    }

## 18. Node: generate a grounded answer

The model receives the user question and the evidence collected by the workflow.

The prompt tells the model not to invent missing facts.

In [ ]:
def generate_node(state: AgentState):
    prompt = f"""
You are a support assistant.

Answer the user's question using only the evidence below.

Rules:
- Do not invent missing facts.
- If the evidence is insufficient, say what information is missing.
- Keep the answer concise.
- Mention the relevant source title or document id when possible.
- Do not reveal hidden reasoning.

USER QUESTION:
{state['question']}

KNOWLEDGE-BASE EVIDENCE:
{state['evidence']}

ORDER INFORMATION:
{state['order_result'] or 'No order lookup was required.'}

FINAL ANSWER:
"""

    answer = llm.invoke(prompt)

    return {
        "answer": answer.strip(),
        "trace": ["generate -> final answer"],
    }

## 19. Conditional routing functions

These functions decide which graph edge should be followed next.

In [ ]:
MAX_REWRITES = 2


def after_retrieval(state: AgentState):
    if state["needs_order"]:
        return "order"

    return "grade"


def after_grade(state: AgentState):
    if state["evidence_ok"]:
        return "generate"

    if state["attempts"] >= MAX_REWRITES:
        return "generate"

    return "rewrite"

## 20. Build the LangGraph workflow

The graph makes the control flow explicit:

**route → retrieve → optional order lookup → grade → rewrite if needed → generate**

Notice that `rewrite` loops back to `retrieve`.

That loop is what makes the retrieval process corrective rather than fixed.

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("route", route_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("order", order_node)
builder.add_node("grade", grade_node)
builder.add_node("rewrite", rewrite_node)
builder.add_node("generate", generate_node)

builder.add_edge(START, "route")
builder.add_edge("route", "retrieve")

builder.add_conditional_edges(
    "retrieve",
    after_retrieval,
    {
        "order": "order",
        "grade": "grade",
    },
)

builder.add_edge("order", "grade")

builder.add_conditional_edges(
    "grade",
    after_grade,
    {
        "generate": "generate",
        "rewrite": "rewrite",
    },
)

builder.add_edge("rewrite", "retrieve")
builder.add_edge("generate", END)

agent = builder.compile()

print("LangGraph workflow compiled.")

### Assignment task — explain the graph

Add a short Markdown explanation of:

- what is stored in state,
- what causes the order tool to run,
- what causes the rewrite loop,
- and what stops the loop.

## 21. Helper function for running the agent

In [ ]:
def run_agent(question):
    initial_state = {
        "question": question,
        "query": question,
        "needs_order": False,
        "order_id": "",
        "evidence": "",
        "order_result": "",
        "evidence_ok": False,
        "attempts": 0,
        "answer": "",
        "trace": [],
    }

    result = agent.invoke(initial_state)

    print("QUESTION")
    print(result["question"])

    print("\nTRACE")
    for step in result["trace"]:
        print("-", step)

    print("\nFINAL ANSWER")
    print(result["answer"])

    return result

## 22. Try the baseline workflow

In [ ]:
result = run_agent(
    "Order 1042 was delivered 20 days ago. Can I still return the laptop?"
)

## 23. Final assignment testing

Test at least **four questions**.

Your tests must include:

1. a normal knowledge-base question,
2. a question containing an order number,
3. a question that uses the additional order you created,
4. a question where your workflow has to rewrite the search query or otherwise correct weak retrieval.

Replace or add to the examples below.

In [ ]:
test_questions = [
    "What is the normal return period for opened electronics?",
    "Order 1042 was delivered 20 days ago. Can I still return the laptop?",
    "How long can a store hold an order for pickup?",
    # TODO: add a question using the new order you created
    # TODO: add a question designed to trigger your corrective retrieval step
]

for question in test_questions:
    print("\n" + "#" * 90)
    run_agent(question)

## 24. Show retrieval evidence for one test

For at least one of your questions, display the retrieved evidence so it is possible to see whether the final answer was grounded.

In [ ]:
# TODO: replace this with one of your final test questions

question = "What is the normal return period for opened electronics?"

docs = retriever.invoke(question)

for i, doc in enumerate(docs, start=1):
    print("=" * 80)
    print("Result", i)
    print("document_id:", doc.metadata["document_id"])
    print("title:", doc.metadata["title"])
    print("category:", doc.metadata["category"])
    print("region:", doc.metadata["region"])
    print()
    print(doc.page_content[:900])

## 25. Reflection

Replace the prompts below with your answers.

### 1. What makes this workflow agentic instead of normal RAG?

Your answer:

### 2. What chunk size and overlap did you choose, and why?

Your answer:

### 3. How does your workflow decide which tool to use?

Your answer:

### 4. How does your evidence check work?

Your answer:

### 5. What happens when retrieval is not good enough?

Your answer:

### 6. What prevents the workflow from looping forever?

Your answer:

### 7. Give one example where the final answer depended on both retrieved policy and structured order information.

Your answer:

# Submission checklist

Before submitting your notebook, make sure you have:

- [ ] loaded the large Module 9 CSV from GitHub
- [ ] compared at least two chunking settings
- [ ] chosen and explained your final chunk size and overlap
- [ ] preserved metadata after chunking
- [ ] created embeddings
- [ ] stored chunks in Chroma
- [ ] tested retrieval separately
- [ ] used at least two tools
- [ ] added at least one order record
- [ ] modified the routing logic
- [ ] improved the evidence check using at least two signals
- [ ] improved the query rewrite
- [ ] used LangGraph
- [ ] kept a maximum rewrite / stop condition
- [ ] tested at least four questions
- [ ] displayed source evidence for at least one result
- [ ] completed the reflection

## Keep the project simple

A clear workflow is more important than a complicated one.

The key idea is:

**State → Decision → Tool → Observation → Check → Retry or Generate → Stop**